<a href="https://colab.research.google.com/github/nithinbharathi-t/AI-integrated-Transcriber/blob/main/AudRAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Install required packages

In [6]:
!pip install librosa gTTS pydub nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 8.2 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.3.1
    Uninstalling click-8.3.1:
      Successfully uninstalled click-8.3.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
typer 0.24.1 requires click>=8.2.1, but you have click 8.1.8 which is incompatible.


## Text to audio

In [ ]:
from gtts import gTTS
from pydub import AudioSegment
import os

def text_to_audio(text, output_file, temp_file):
    if not output_file.endswith(".wav"):
        final_filename = f"{output_file}.wav"
    else:
        final_filename = output_file

    temp_mp3 = temp_file + ".mp3"

    try:
        tts = gTTS(text=text, lang='en')
        tts.save(temp_mp3)
        audio = AudioSegment.from_mp3(temp_mp3)
        audio.export(final_filename, format="wav")

        print(f"Successfully saved: {final_filename}")

    finally:
        if os.path.exists(temp_mp3):
            os.remove(temp_mp3)
text_to_audio("Hello! This function is now ready to use.", "my_speech_file", )

## No cost tts

In [7]:
!apt-get install -y espeak-ng && pip install pyttsx3

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
espeak-ng is already the newest version (1.50+dfsg-10ubuntu0.1).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.


In [8]:
import pyttsx3
import os

engine = pyttsx3.init()

engine.setProperty('rate', 160)
engine.setProperty('volume', 1.0)

def text_to_audio(text, output_file):
    if not output_file.endswith(".wav"):
        final_filename = f"{output_file}.wav"
    else:
        final_filename = output_file
    try:
        engine.save_to_file(text, final_filename)
        engine.runAndWait()
    except Exception as e:
        print(f"Error saving {final_filename}: {e}")

# text_to_audio("Hello! This function is now offline and fast.", "test_output")

## Audio to Mel-Spectrum

In [10]:
import librosa
import librosa.display
import numpy as np
import matplotlib.pyplot as plt

def audio_to_spectrogram(file_path, vec, output_image):
    y, sr = librosa.load(file_path, sr=22050)
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
    S_dB = librosa.power_to_db(S, ref=np.max)

    print(f"Spectrogram Shape: {S_dB.shape}")
    np.save(f"{vec}.npy", S_dB)
    plt.figure(figsize=(10, 4))
    librosa.display.specshow(S_dB, sr=sr, x_axis='time', y_axis='mel')
    plt.colorbar(format='%+2.0f dB')
    plt.title('Mel-Spectrogram')
    plt.tight_layout()
    plt.savefig(f"{output_image}.png")
    plt.close()

# audio_to_spectrogram("my_speech_file.wav", "spec_vector", "mel_spectrogram")

## Mel-Spectrum to Audio

In [11]:
import librosa
import soundfile as sf
import numpy as np

def spectrogram_to_audio(spec_path, output_name, sr=22050, gain=1.5):
    S_dB = np.load(spec_path)
    S = librosa.db_to_power(S_dB)
    y_reconstructed = librosa.feature.inverse.mel_to_audio(S, sr=sr)
    y_reconstructed = y_reconstructed * gain
    y_reconstructed = np.clip(y_reconstructed, -1.0, 1.0)
    sf.write(f"{output_name}.wav", y_reconstructed, sr)
    print(f"Reconstructed audio saved as {output_name}.wav with gain {gain}")

# spectrogram_to_audio("spec_vector.npy", "output_from_spec", gain=4.0)

## Pre-setting environment

In [16]:
!rm -rf sentence_chunks/
!rm -rf audio_llm_dataset/

## Text data chunking

In [ ]:
import nltk # You may need to run: pip install nltk
import os

# Download the sentence tokenizer
nltk.download('punkt')
nltk.download('punkt_tab')

def chunk_by_sentence(input_file, output_folder, sentences_per_chunk=5):
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    with open(input_file, 'r', encoding='utf-8') as f:
        text = f.read()

    # Split text into a list of sentences
    sentences = nltk.tokenize.sent_tokenize(text)

    for i in range(0, len(sentences), sentences_per_chunk):
        chunk = sentences[i:i + sentences_per_chunk]
        chunk_content = " ".join(chunk)

        output_path = os.path.join(output_folder, f"sent_chunk_{i // sentences_per_chunk}.txt")
        with open(output_path, 'w', encoding='utf-8') as out_f:
            out_f.write(chunk_content)

# Usage
chunk_by_sentence("big.txt", "sentence_chunks")

## Non-parallel

In [15]:
import os
import numpy as np

# Importing your specific modules
# from text2audio import text_to_audio
# from audio2melspectrum import audio_to_spectrogram
# from melspectrum2audio import spectrogram_to_audio

def load_chunks_from_file(file_path, words_per_chunk=50):
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            words = f.read().split()
        chunks = [
            " ".join(words[i : i + words_per_chunk])
            for i in range(0, len(words), words_per_chunk)
        ]
        print(f"Successfully loaded {len(chunks)} samples from '{file_path}'.")
        return chunks

    except FileNotFoundError:
        print("Error: The text file was not found.")
        return []

FILE_PATH = "big.txt"
TEXT_SAMPLES = load_chunks_from_file(FILE_PATH, words_per_chunk=150)

# --- CONFIGURATION ---
DATASET_DIR = "audio_llm_dataset"
# TEXT_SAMPLES = [
#     "The quick brown fox jumps over the lazy dog.",
#     "Artificial intelligence is transforming the world of audio.",
#     "Generative models can now synthesize realistic human speech.",
#     "Training a transformer on mel spectrograms requires significant data.",
#     "Consistency in sample rate is crucial for high-quality reconstruction."
# ]

def setup_directories():
    """Creates the necessary folder structure for the dataset."""
    folders = ["wavs", "mels", "plots", "reconstructed"]
    for folder in folders:
        os.makedirs(os.path.join(DATASET_DIR, folder), exist_ok=True)
    print(f"-> Directory structure ready at: {DATASET_DIR}")

def generate_dataset():
    setup_directories()

    print(f"Starting dataset generation for {len(TEXT_SAMPLES)} samples...")

    for i, text in enumerate(TEXT_SAMPLES):
        sample_id = f"sample_{i:04d}"

        # Define paths
        wav_path = os.path.join(DATASET_DIR, "wavs", sample_id) # text2audio adds .wav
        mel_vec_path = os.path.join(DATASET_DIR, "mels", sample_id) # audio2mel adds .npy
        mel_img_path = os.path.join(DATASET_DIR, "plots", f"{sample_id}.png")
        recon_path = os.path.join(DATASET_DIR, "reconstructed", sample_id) # mel2audio adds .wav

        try:
            text_to_audio(text, wav_path)
            audio_to_spectrogram(f"{wav_path}.wav", mel_vec_path, mel_img_path)
            spectrogram_to_audio(f"{mel_vec_path}.npy", recon_path, sr=22050)

            print(f"[SUCCESS] Processed {sample_id}")

        except Exception as e:
            print(f"[ERROR] Failed to process {sample_id}: {e}")

    save_metadata()

def save_metadata():
    metadata_path = os.path.join(DATASET_DIR, "metadata.csv")
    with open(metadata_path, "w") as f:
        f.write("id|text|mel_path\n")
        for i, text in enumerate(TEXT_SAMPLES):
            sample_id = f"sample_{i:04d}"
            f.write(f"{sample_id}|{text}|mels/{sample_id}.npy\n")
    print(f"-> Metadata saved to {metadata_path}")

if __name__ == "__main__":
    generate_dataset()

Successfully loaded 7305 samples from 'big.txt'.
-> Directory structure ready at: audio_llm_dataset
Starting dataset generation for 7305 samples...
Audio saved to audio_llm_dataset/wavs/sample_0000.wav


KeyboardInterrupt: 

## First parallet try

In [12]:
import os
import numpy as np
from concurrent.futures import ProcessPoolExecutor, as_completed

# Assuming these are your local modules
# from text2audio import text_to_audio
# from audio2melspectrum import audio_to_spectrogram
# from melspectrum2audio import spectrogram_to_audio

DATASET_DIR = "audio_llm_dataset"

def load_chunks_from_file(file_path, words_per_chunk=150):
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            words = f.read().split()
        chunks = [" ".join(words[i : i + words_per_chunk]) for i in range(0, len(words), words_per_chunk)]
        print(f"Successfully loaded {len(chunks)} samples.")
        return chunks
    except FileNotFoundError:
        print("Error: The text file was not found.")
        return []

def setup_directories():
    """Creates the necessary folder structure."""
    folders = ["wavs", "mels", "plots", "reconstructed"]
    for folder in folders:
        os.makedirs(os.path.join(DATASET_DIR, folder), exist_ok=True)
    print(f"-> Directory structure ready at: {DATASET_DIR}")

def process_single_sample(args):
    """
    Function to process a single item.
    Multiprocessing works best when logic is encapsulated in a single function.
    """
    i, text = args
    sample_id = f"sample_{i:04d}"

    wav_path = os.path.join(DATASET_DIR, "wavs", sample_id)
    mel_vec_path = os.path.join(DATASET_DIR, "mels", sample_id)
    mel_img_path = os.path.join(DATASET_DIR, "plots", f"{sample_id}.png")
    recon_path = os.path.join(DATASET_DIR, "reconstructed", sample_id)

    try:
        # 1. Text to Audio
        text_to_audio(text, wav_path)

        # 2. Audio to Spectrogram
        audio_to_spectrogram(f"{wav_path}.wav", mel_vec_path, mel_img_path)

        # 3. Spectrogram to Audio (Reconstruction)
        spectrogram_to_audio(f"{mel_vec_path}.npy", recon_path, sr=22050)

        return f"[SUCCESS] Processed {sample_id}"
    except Exception as e:
        return f"[ERROR] Failed to process {sample_id}: {e}"

def save_metadata(samples):
    metadata_path = os.path.join(DATASET_DIR, "metadata.csv")
    with open(metadata_path, "w") as f:
        f.write("id|text|mel_path\n")
        for i, text in enumerate(samples):
            sample_id = f"sample_{i:04d}"
            f.write(f"{sample_id}|{text}|mels/{sample_id}.npy\n")
    print(f"-> Metadata saved to {metadata_path}")

def generate_dataset_parallel(samples, max_workers=4):
    """
    Runs the generation process in parallel.
    max_workers: Number of CPU cores to use.
    Set this to 4, 8, or 16 depending on your CPU.
    """
    setup_directories()

    # Prepare arguments for the worker function
    tasks = [(i, text) for i, text in enumerate(samples)]

    print(f"Starting parallel generation with {max_workers} workers...")

    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        # Map the function to the tasks
        futures = [executor.submit(process_single_sample, task) for task in tasks]

        for future in as_completed(futures):
            print(future.result())

    save_metadata(samples)

if __name__ == "__main__":
    # 1. Load data
    FILE_PATH = "big.txt"
    TEXT_SAMPLES = load_chunks_from_file(FILE_PATH)

    if TEXT_SAMPLES:
        # 2. Run parallel generation
        # Adjust max_workers based on your CPU (e.g., 4, 8, or 12)
        generate_dataset_parallel(TEXT_SAMPLES, max_workers=8)

Successfully loaded 7305 samples.
-> Directory structure ready at: audio_llm_dataset
Starting parallel generation with 8 workers...
Successfully saved: audio_llm_dataset/wavs/sample_0004.wav
Spectrogram Shape: (128, 2660)
[ERROR] Failed to process sample_0007: Decoding failed. ffmpeg returned error code: 1

Output from ffmpeg/avlib:

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enabl

KeyboardInterrupt: 

## Altered temp file name

In [14]:
import os
import numpy as np
from concurrent.futures import ProcessPoolExecutor, as_completed

# Assuming these are your local modules
# from text2audio import text_to_audio
# from audio2melspectrum import audio_to_spectrogram
# from melspectrum2audio import spectrogram_to_audio

DATASET_DIR = "audio_llm_dataset"

def load_chunks_from_file(file_path, words_per_chunk=150):
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            words = f.read().split()
        chunks = [" ".join(words[i : i + words_per_chunk]) for i in range(0, len(words), words_per_chunk)]
        print(f"Successfully loaded {len(chunks)} samples.")
        return chunks
    except FileNotFoundError:
        print("Error: The text file was not found.")
        return []

def setup_directories():
    """Creates the necessary folder structure."""
    folders = ["wavs", "mels", "plots", "reconstructed", "temp"] # Added a temp folder
    for folder in folders:
        os.makedirs(os.path.join(DATASET_DIR, folder), exist_ok=True)
    print(f"-> Directory structure ready at: {DATASET_DIR}")

def process_single_sample(args):
    """
    Function to process a single item.
    """
    i, text = args
    sample_id = f"sample_{i:04d}"

    # Standard output paths
    wav_path = os.path.join(DATASET_DIR, "wavs", sample_id)
    mel_vec_path = os.path.join(DATASET_DIR, "mels", sample_id)
    mel_img_path = os.path.join(DATASET_DIR, "plots", f"{sample_id}.png")
    recon_path = os.path.join(DATASET_DIR, "reconstructed", sample_id)

    # UNIQUE TEMPORARY FILE PATH
    # This ensures no worker ever collides with another
    unique_temp_file = os.path.join(DATASET_DIR, "temp", f"temp_{sample_id}.wav")

    try:
        # 1. Text to Audio
        # CRITICAL: You must update text_to_audio to use unique_temp_file
        # instead of a hardcoded "temp.wav" if it uses one internally.
        text_to_audio(text, wav_path)

        # 2. Audio to Spectrogram
        audio_to_spectrogram(f"{wav_path}.wav", mel_vec_path, mel_img_path)

        # 3. Spectrogram to Audio (Reconstruction)
        spectrogram_to_audio(f"{mel_vec_path}.npy", recon_path, sr=22050)

        return f"[SUCCESS] Processed {sample_id}"

    except Exception as e:
        return f"[ERROR] Failed to process {sample_id}: {e}"

    finally:
        # Cleanup: Delete the unique temp file after the worker finishes (success or fail)
        if os.path.exists(unique_temp_file):
            try:
                os.remove(unique_temp_file)
            except OSError:
                pass # Fail silently if it was already cleaned up

def save_metadata(samples):
    metadata_path = os.path.join(DATASET_DIR, "metadata.csv")
    with open(metadata_path, "w") as f:
        f.write("id|text|mel_path\n")
        for i, text in enumerate(samples):
            sample_id = f"sample_{i:04d}"
            f.write(f"{sample_id}|{text}|mels/{sample_id}.npy\n")
    print(f"-> Metadata saved to {metadata_path}")

def generate_dataset_parallel(samples, max_workers=4):
    setup_directories()

    tasks = [(i, text) for i, text in enumerate(samples)]
    print(f"Starting parallel generation with {max_workers} workers...")

    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(process_single_sample, task) for task in tasks]
        for future in as_completed(futures):
            print(future.result())

    # Optional: Clean up the empty temp directory at the very end
    temp_dir = os.path.join(DATASET_DIR, "temp")
    if os.path.exists(temp_dir) and not os.listdir(temp_dir):
        os.rmdir(temp_dir)

    save_metadata(samples)

if __name__ == "__main__":
    FILE_PATH = "big.txt"
    # TEXT_SAMPLES = load_chunks_from_file(FILE_PATH)
    TEXT_SAMPLES = [
    "The quick brown fox jumps over the lazy dog.",
    "Artificial intelligence is transforming the world of audio.",
    "Generative models can now synthesize realistic human speech.",
    "Training a transformer on mel spectrograms requires significant data.",
    "Consistency in sample rate is crucial for high-quality reconstruction."
]

    if TEXT_SAMPLES:
        generate_dataset_parallel(TEXT_SAMPLES, max_workers=8)

-> Directory structure ready at: audio_llm_dataset
Starting parallel generation with 8 workers...


Process ForkProcess-15:
Process ForkProcess-14:
Process ForkProcess-16:
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
  File "/usr/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/usr/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/usr/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/usr/lib/python3.12/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/lib/python3.12/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/lib/python3.12/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/lib/python3.12/concurrent/futures/process.py", line 252, in _process_worker
    call_item = call_queue.get(block=True)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^


KeyboardInterrupt: 

In [ ]:
from google.colab import output

def play_alert():
  # This uses a standard notification 'ping' sound URL
  # You can replace this URL with your own .wav or .mp3 link
  url = "https://www.gstatic.com/dictionary/static/sounds/oxford/bell--_gb_1.mp3"

  output.eval_js(f'new Audio("{url}").play()')

# --- Usage in your long program ---
import time

print("Starting huge task...")
time.sleep(5) # Simulating a long task

print("Task Complete!")
play_alert()

## Training the model